# PABPC1 Stage 3 — Rebuild the missing RRM2–RRM3 linker
**Purpose:** Test whether residues 176–190 (`RKEREAELGARAKEF`) can be rebuilt between the 4F02-derived RRM1/2 and the retained RRM3/4 coordinates. The starting Cα endpoint distance is **40.76 Å**, so this is a challenging reconstruction. **This is not a validated final structure and must not be sent directly to MM/GBSA.**

The Colab notebook uses [MODELLER's documented missing-residues workflow](https://salilab.org/modeller/resources/missing_residues.html). MODELLER requires a license, free for eligible academic users. A link to register is [here](https://salilab.org/modeller/registration.html). Do not put your license key in a file or notebook saved to Drive.
**Fix (2026-11-22):** Stage 4 explicitly selects the long A176–A190 linker because MODELLER's default finder may skip long loops. This update also chooses Conda Python and screens only loop-refined BL models.


### 1 — Install Conda (first cell; runtime will restart)
Run this cell with the ▶ icon. Google Colab will restart the runtime, then continue with Step 2.

In [ ]:
!pip -q install condacolab
import condacolab
condacolab.install()

### 2 — Enter the MODELLER license key and install MODELLER
Get your own licence from the official registration page above. The key is entered through a hidden prompt and will not be saved into this notebook. This installation uses the documented `salilab` Conda channel. If the installation fails, copy the **error message without the license key**.

In [ ]:
import os, getpass, subprocess
license_key = getpass.getpass("MODELLER license key (hidden): ").strip()
if not license_key:
    raise ValueError("MODELLER requires a valid license key.")
os.environ['KEY_MODELLER'] = license_key
os.environ['KEY_MODELLER10v8'] = license_key
cmd=['conda','install','-y','-c','salilab','-c','conda-forge','modeller']
subprocess.run(cmd,check=True)
import modeller
print('MODELLER imported successfully.')

### 3 — Upload the prepared inputs ZIP
Upload `PABPC1_Linker_Modeling_Kit.zip` from the companion download. No need to upload your PDB again: it is inside the ZIP.

In [ ]:
from google.colab import files
import zipfile
from pathlib import Path
uploaded=files.upload()
names=[n for n in uploaded if n.endswith('.zip')]
if len(names)!=1:
    raise ValueError('Upload exactly one PABPC1_Linker_Modeling_Kit.zip file.')
root=Path('/content/PABPC1_Linker_Modeling_Kit')
root.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(names[0]) as z:
    if any(Path(x).is_absolute() or '..' in Path(x).parts for x in z.namelist()):
        raise ValueError('Unsafe ZIP path')
    z.extractall(root)
required=['PABPC1_template.pdb','PABPC1_alignment.ali','run_modeller.py','check_linker_models.py']
assert all((root/x).exists() for x in required), 'ZIP inputs missing'
print('Inputs loaded. Files:',len(list(root.iterdir())))

### 4 — Rebuild the missing linker (explicit long-loop selection)
The default MODELLER loop finder can skip long segments. This cell explicitly selects PABPC1 residues A176–A190 and renumbers the 398-residue target from 11. It uses Conda Python, not the Colab runtime Python. Run one pilot with four loop candidates before scaling.


In [ ]:
import subprocess, os, sys
from pathlib import Path

root = Path('/content/PABPC1_Linker_Modeling_Kit')
assert root.is_dir(), 'Run the ZIP upload cell before modelling.'
script = root / 'run_modeller.py'
backup = root / 'run_modeller_BEFORE_LONG_LOOP_FIX.py'
if not backup.exists():
    backup.write_text(script.read_text())
script.write_text("\"\"\"Build missing PABPC1 residues 176-190 (pilot; not MMGBSA ready).\"\"\"\nfrom modeller import Environ, log, Selection\nfrom modeller.automodel import LoopModel, refine, assess\n\nlog.verbose()\nenv = Environ()\nenv.io.atom_files_directory = ['.']\n\nclass ExplicitPABPC1Linker(LoopModel):\n    def special_patches(self, aln):\n        # Input protein numbering is A11-A408, not A1-A398.\n        self.rename_segments(segment_ids=['A'], renumber_residues=[11])\n\n    def select_loop_atoms(self):\n        # Force refinement for the 15-residue linker; MODELLER's\n        # default loop detection can skip longer inserted segments.\n        print('Explicit loop selection: A176-A190')\n        return Selection(self.residue_range('176:A', '190:A'))\n\nmodel = ExplicitPABPC1Linker(\n    env,\n    alnfile='PABPC1_alignment.ali',\n    knowns='PABPC1_template',\n    sequence='PABPC1_rebuilt',\n    assess_methods=(assess.DOPE,),\n    loop_assess_methods=(assess.DOPE,),\n)\nmodel.starting_model = 1\nmodel.ending_model = 1\nmodel.md_level = None\nmodel.loop.starting_model = 1\nmodel.loop.ending_model = 4\nmodel.loop.md_level = refine.fast\nmodel.make()\nprint('MODELLER pilot completed. Screen candidates before MMGBSA.')\n")

base = subprocess.check_output(['conda', 'info', '--base'], text=True).strip()
modeller_python = Path(base) / 'bin' / 'python'
print('Using:', modeller_python)
print('MODELLER will explicitly refine protein A176-A190 (one pilot + four loop conformations).')
result = subprocess.run(
    [str(modeller_python), '-u', str(script)],
    cwd=str(root),
    env=os.environ.copy()
)
print('Return code:', result.returncode)
if result.returncode:
    raise RuntimeError('MODELLER failed; inspect the error above. Do not proceed to MMGBSA.')
loop_models = sorted(root.glob('PABPC1_rebuilt.BL*.pdb'))
print('Loop-refined candidate PDBs:', len(loop_models))
if not loop_models:
    raise RuntimeError('MODELLER exited but wrote no BL loop-refined models. Inspect logs.')
print('Pilot completed. Candidate structures still need independent geometry checks.')


### 5 — Screen models for gross errors and restore the reference A₉ RNA
This cell aligns each candidate to your original RRM1/2 frame, appends the A₉ RNA for **inspection only**, and reports backbone bond distances and severe clashes. **Even a PASS is not proof of the correct RRM3/4 pose.**

In [ ]:
!pip -q install biopython scipy
# Screen only the BL (loop-refined) outputs; avoid unrefined initial B models.
checker = root / 'check_linker_models.py'
checker_text = checker.read_text()
checker_text = checker_text.replace(
    "sorted(ROOT.glob('PABPC1_rebuilt*.pdb'))",
    "sorted(ROOT.glob('PABPC1_rebuilt.BL*.pdb'))"
)
checker.write_text(checker_text)
r=subprocess.run([sys.executable,'check_linker_models.py'],cwd=str(root))
if r.returncode:
    raise RuntimeError('Screening failed; see the error above.')
import pandas as pd
from IPython.display import display
screen=pd.read_csv(root/'model_preflight_screen.csv')
display(screen)

### 6 — Download results and share the screening CSV for scientific review
The ZIP contains the geometry table, raw candidate models, and PDBs clearly marked `REVIEW_ONLY`. **Do not load any of these into your MD/MMGBSA pipeline before the domain orientations and linker geometry are reviewed.**

In [ ]:
import zipfile
out=Path('/content/PABPC1_LINKER_CANDIDATE_RESULTS_REVIEW_ONLY.zip')
with zipfile.ZipFile(out,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for path in sorted(root.iterdir()):
        if path.is_file() and not path.name.endswith('.log'):
            z.write(path,arcname=path.name)
files.download(str(out))
print('Share model_preflight_screen.csv and any errors with your mentor or here for the next structural decision.')